# 02 VAR Model Training

This notebook loads the merged monthly dataset, checks stationarity with the Augmented Dickey-Fuller test, differences the series if needed, runs Granger causality tests, fits a Vector Autoregression (VAR) model, and saves the trained model.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 6)

ARTIFACT_DIR = Path("artifacts")
DATA_PATH = ARTIFACT_DIR / "merged_monthly_prices.csv"
MODEL_PATH = ARTIFACT_DIR / "shockwave_var_model.pkl"

In [ ]:
merged_df = pd.read_csv(DATA_PATH, parse_dates=["Date"], index_col="Date")
merged_df = merged_df.asfreq("MS")
merged_df.head()

## Why stationarity matters

A VAR model assumes the joint mean and covariance structure is stable over time. If the series has a unit root, lag coefficients can become misleading. The Augmented Dickey-Fuller (ADF) test checks whether the null hypothesis of a unit root can be rejected.

If `p-value > 0.05`, we treat the series as non-stationary and difference it. First differencing replaces `y_t` with `y_t - y_(t-1)`, which removes many trend-driven non-stationary patterns.

In [ ]:
def adf_summary(series: pd.Series, series_name: str) -> dict[str, float]:
    result = adfuller(series.dropna(), autolag="AIC")
    summary = {
        "series": series_name,
        "adf_statistic": result[0],
        "p_value": result[1],
        "n_lags": result[2],
        "n_obs": result[3],
    }
    return summary


stationarity_report = pd.DataFrame(
    [adf_summary(merged_df[col], col) for col in merged_df.columns]
)
stationarity_report

In [ ]:
needs_difference = (stationarity_report["p_value"] > 0.05).any()

if needs_difference:
    model_df = merged_df.diff().dropna()
    print("At least one series is non-stationary. Using first-differenced data for VAR.")
else:
    model_df = merged_df.copy()
    print("Both series pass the stationarity check. Using level data for VAR.")

model_df.head()

In [ ]:
post_diff_stationarity_report = pd.DataFrame(
    [adf_summary(model_df[col], col) for col in model_df.columns]
)
post_diff_stationarity_report

## Why Granger causality is used here

Granger causality does not prove physical causation. It tests whether lagged values of one series add statistically significant predictive information for another series after accounting for the target's own lag history.

For SHOCKWAVE, this is a useful first screen: if lagged `EIA_Price` consistently improves prediction of `DOEB_Volume`, then external market signals may be valuable early-warning features.

In [ ]:
max_test_lag = min(6, max(2, len(model_df) // 5))
granger_input = model_df[["DOEB_Volume", "EIA_Price"]]

granger_results = grangercausalitytests(granger_input, maxlag=max_test_lag, verbose=True)

granger_summary = []
for lag, test_result in granger_results.items():
    ssr_ftest_pvalue = test_result[0]["ssr_ftest"][1]
    granger_summary.append({"lag": lag, "ssr_ftest_pvalue": ssr_ftest_pvalue})

granger_summary_df = pd.DataFrame(granger_summary)
granger_summary_df

Interpretation guideline: if one or more lag orders produce `p < 0.05`, then lagged `EIA_Price` contains statistically useful information for forecasting `DOEB_Volume`.

## VAR lag-order logic

A VAR(p) model uses the last `p` observations of every variable in the system. Too few lags underfit the dependency structure; too many lags overfit and waste degrees of freedom.

AIC and BIC balance fit against complexity. AIC usually prefers richer models, while BIC is stricter. In hackathon settings, a common pragmatic rule is to inspect both and select the lag that is stable and not excessively large for the sample size.

In [ ]:
var_model = VAR(model_df)
max_var_lag = min(12, max(2, len(model_df) // 4))
lag_selection = var_model.select_order(maxlags=max_var_lag)

print(lag_selection.summary())

In [ ]:
selected_lag = lag_selection.aic
if selected_lag is None:
    selected_lag = lag_selection.bic
if selected_lag is None:
    raise ValueError("Unable to determine an optimal lag from AIC/BIC.")

selected_lag = max(1, int(selected_lag))
print(f"Selected VAR lag order: {selected_lag}")

var_results = var_model.fit(selected_lag)
print(var_results.summary())

In [ ]:
forecast_horizon = 6
forecast_values = var_results.forecast(y=model_df.values[-selected_lag:], steps=forecast_horizon)
forecast_index = pd.date_range(model_df.index[-1] + pd.offsets.MonthBegin(), periods=forecast_horizon, freq="MS")
forecast_df = pd.DataFrame(forecast_values, index=forecast_index, columns=model_df.columns)

display(forecast_df)

ax = model_df.plot(title="Stationary VAR Training Data")
forecast_df.plot(ax=ax, style="--")
plt.show()

In [ ]:
model_bundle = {
    "var_results": var_results,
    "selected_lag": selected_lag,
    "is_differenced": needs_difference,
    "train_columns": list(model_df.columns),
    "train_index_end": model_df.index.max(),
    "granger_summary": granger_summary_df,
}

joblib.dump(model_bundle, MODEL_PATH)
print(f"Saved VAR model bundle to: {MODEL_PATH.resolve()}")